## **Revision 05 -- Era and temporal sensitivity of the SMM model (24-hour prediction point)**

**Purpose.** Check whether the model learns *when* an admission happened (calendar era, ICD-9 vs ICD-10 coding) instead of
the severity of the patient, and how it behaves when it is applied to patients from a later period than the one it was
developed on. It addresses **R1-3, R2-6, R2-17, R2-25** and, partly, **R3-7** (plan tasks **T47**, **T30**). It reads the
tables built by `sql/05_revision_audit`, `sql/06_revision_labels` and `sql/07_revision_predictors` and does **not** modify
the database or any previous notebook.

**Design.** Same population, predictors, outer partitions (same seed, patient-grouped, 5 folds x 3 repeats), inner tuning,
explicit stacking and recalibration as `02_grouped_validation_smm.ipynb` / `04_...`.

| Analysis | What is done |
|---|---|
| **A. Performance by group** | The out-of-fold predictions of `main` (a replicate of notebooks 02/04) are evaluated separately in each `anchor_year_group` (2008-2010 ... 2020-2022) and in each ICD coding version (ICD-9 only, mixed, ICD-10 only): AUROC, AUPRC, O/E and calibration slope with patient-level cluster-bootstrap intervals. |
| **B. Rolling-origin temporal validation** | For each later group, every learner is tuned, stacked and recalibrated **only** on the earlier groups and scored on that group. Patients with admissions in the test group are removed from the training block, so no patient is on both sides. A fit is skipped (and printed as skipped) when the training block has fewer than `MIN_TRAIN_EVENTS` or the test block fewer than `MIN_TEST_EVENTS` events. |
| **C. ICD-9 to ICD-10 transition** | The same procedure, training on ICD-9-only admissions and testing on ICD-10-only admissions (same patient exclusion). |
| **D. Refits without era proxies** | `main` against (i) `main_noadmin`: age, hemoglobin, creatinine and the creatinine indicator only, without the five administrative categorical variables (ethnicity, insurance, language, admission type and location, whose categories and use change over time), and (ii) `main_plus_era`: `main` plus the anchor-year group, as a **diagnostic only** (it is not a deployable predictor). Same splits and same patient resamples, so the differences are paired. |

**What the results can and cannot show.** `anchor_year_group` is a three-year group assigned by MIMIC-IV to each **patient**,
not an exact admission date: a repeat admission years later keeps the patient's group, and the group is therefore an
approximate calendar marker. Because it is fixed per patient, every patient lies wholly in one calendar block. The ICD-9 to
ICD-10 change (October 2015) falls inside the 2014-2016 group, so era and coding version are partly confounded. The temporal validation is exploratory: it is one institution, with few events in the early groups, and it is
**not** an external validation.

**Data-use rules.** Only aggregate results are printed; the only file written is `outputs_era_temporal.txt` (aggregate text,
listed in `.gitignore` through `outputs_*.txt`). Calendar groups are read into memory for the analysis and never printed or
saved at patient level.

**Run order.** Run all cells top to bottom with `QUICK = False`. Cells in section 7 are separate so that a finished analysis
is kept if a later one is interrupted. Time: three nested cross-validations (about three times the main run of notebook 02)
plus five short temporal fits. Set `QUICK = True` first for a smoke test (in that mode most temporal fits will be skipped).

### **1. Libraries, connection and settings**

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/revision/ -> project root is two levels up
sys.path.append(os.path.abspath(os.path.join("..", "..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False        # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- settings specific to notebook 05
ERA_ORDER    = ["2008 - 2010", "2011 - 2013", "2014 - 2016", "2017 - 2019", "2020 - 2022"]   # anchor_year_group, MIMIC-IV
MIN_TRAIN_EVENTS = 40       # a temporal fit is skipped (and reported as skipped) below this number of training events
MIN_TEST_EVENTS  = 15       # ... or below this number of test events
MIN_SLOPE_EVENTS = 20       # calibration slope is shown only for groups with at least this many events
EVAL_MODELS  = ["LR", "XGB", "Stack"]

QUICK=False  outer=5x3  inner=3  B=1000  trees=300


### **2. Load the predictor table, the dictionary and the calendar information**

`predictors_24h_v2` has one row per obstetric hospitalization. `anchor_year_group` comes from `mimiciv.patients` and the ICD coding version from `mimiciv.cohort_obstetric_v2` (`has_icd9_dx`, `has_icd10_dx`). Only counts are printed.

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()
print("Dictionary roles ->", {"main": VAR_MAIN, "missingness indicator in main model": IND_MAIN})

# Plausibility limits: out-of-range values become missing (decision D16)
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

for v in set(IND_MAIN):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["y"] = df["smm_any"].astype(int)

# ---- calendar information (aggregated groups only; it never leaves this notebook)
era = pd.read_sql("SELECT subject_id, anchor_year_group FROM mimiciv.patients", engine).drop_duplicates("subject_id")
era = era.rename(columns={"anchor_year_group": "era"})
df = df.merge(era, on="subject_id", how="left")

ver = pd.read_sql("SELECT hadm_id, has_icd9_dx, has_icd10_dx FROM mimiciv.cohort_obstetric_v2", engine).drop_duplicates("hadm_id")
d9 = ver["has_icd9_dx"].fillna(False).astype(bool); d10 = ver["has_icd10_dx"].fillna(False).astype(bool)
ver["icd_ver"] = np.select([d9 & ~d10, d10 & ~d9, d9 & d10], ["ICD-9 only", "ICD-10 only", "mixed"], default="none")
df = df.merge(ver[["hadm_id", "icd_ver"]], on="hadm_id", how="left")
df["icd_ver"] = df["icd_ver"].fillna("none")

unknown = sorted(set(df["era"].dropna().unique()) - set(ERA_ORDER))
print("anchor_year_group values not in ERA_ORDER:", unknown if unknown else "none")

elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible deliveries with missing era:", int(elig["era"].isna().sum()))

Dictionary roles -> {'main': ['hemoglobin', 'creatinine'], 'missingness indicator in main model': ['creatinine']}
Values set to missing by plausibility limits (count per variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
anchor_year_group values not in ERA_ORDER: none
eligible_deliveries: 19383  (expected 19383)  [OK]
smm: 445  (expected 445)  [OK]
deaths: 3  (expected 3)  [OK]
Eligible deliveries with missing era: 0


### **3. Analysis datasets**

| Name | Predictors |
|---|---|
| `main` | age, hemoglobin, creatinine, creatinine missingness indicator, five categorical admission variables (replicates notebooks 02/04) |
| `main_noadmin` | age, hemoglobin, creatinine, creatinine missingness indicator (no administrative categorical variables) |
| `main_plus_era` | `main` + anchor-year group (diagnostic only) |

All three use the same eligible delivery hospitalizations in the same order, which is what makes the comparison paired.

In [3]:
def make_spec(name):
    """main = replicate of notebooks 02/04; main_noadmin = without the five administrative categorical variables;
    main_plus_era = main plus the anchor-year group (diagnostic only, not a deployable predictor)."""
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    cat = [c for c in CAT_COLS if c in d.columns]
    if name == "main":
        pass
    elif name == "main_noadmin":
        cat = []
    elif name == "main_plus_era":
        cat = cat + ["era"]
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label="y")

for nm in ["main", "main_noadmin", "main_plus_era"]:
    sp = make_spec(nm); d = sp["data"]
    print(f"{nm:14s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={sp['cat']}")

main           admissions= 19383  patients= 13710  events= 445  numeric=3  binary=1  categorical=['ethnicity', 'insurance', 'language', 'admission_type', 'admission_location']
main_noadmin   admissions= 19383  patients= 13710  events= 445  numeric=3  binary=1  categorical=[]
main_plus_era  admissions= 19383  patients= 13710  events= 445  numeric=3  binary=1  categorical=['ethnicity', 'insurance', 'language', 'admission_type', 'admission_location', 'era']


### **4. Core functions (preprocessing, learners, nested tuning, stacking, calibration)**

Everything that learns from data is inside the training portion of each split:

* `Winsor` clips each numeric column to the 0.5th-99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped CV** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [4]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # Widened after the first run, in which several learners chose the edge of the original grid
    # (ET depth 8 of {4, 8}; RF depth 4 of {4, 8}; XGB depth 2 of {2, 4}; no class weight ever chosen).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=""):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

### **5. Metrics, cluster bootstrap and paired differences**

All metrics are computed on the pooled out-of-fold predictions of each repeat and then averaged over repeats.
The bootstrap resamples **patients** (all admissions of a patient move together) and recomputes every metric with
patient-count weights, so intervals reflect the repeated-admission structure. Differences against logistic
regression use the **same** bootstrap samples (paired).

In [5]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

### **6. Era-specific functions**

Performance by group, temporal fits and paired refits. All metrics use patient-count weights, so every interval is a patient-level cluster bootstrap.

In [6]:
def perf(y, p, w):
    """Discrimination and calibration of one set of predictions with patient-count weights."""
    pos = np.sum(w * y); neg = np.sum(w * (1 - y))
    nan = dict(auroc=np.nan, auprc=np.nan, oe=np.nan, slope=np.nan, citl=np.nan)
    if pos < 1 or neg < 1:
        return nan
    z = logit(p)
    return dict(auroc=roc_auc_score(y, p, sample_weight=w), auprc=average_precision_score(y, p, sample_weight=w),
                oe=float(pos / np.sum(w * p)), citl=citl(y, z, w),
                slope=calib_slope(y, z, w) if pos >= MIN_SLOPE_EVENTS else np.nan)

METS = ("auroc", "auprc", "oe", "slope", "citl")

def _ci(arr, d=3):
    a = np.asarray(arr, float); a = a[np.isfinite(a)]
    if len(a) < 20:
        return ""
    lo, hi = np.percentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def _fmt(v, d=3):
    return "n/a" if v is None or not np.isfinite(v) else f"{v:.{d}f}"

def cluster_weights(g, B, seed):
    """Yield B patient-level bootstrap weight vectors (all admissions of a patient move together)."""
    ug, inv = np.unique(g, return_inverse=True)
    rng = np.random.default_rng(seed)
    for _ in range(B):
        yield np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)

# ---- 1. performance by era and ICD version, from the out-of-fold predictions of the main analysis
def group_performance(res, groups, B, seed):
    y = res["y"].astype(float); g = res["g"]
    def one(w):
        out = {}
        for gname, mask in groups.items():
            for m in EVAL_MODELS:
                acc = [perf(y[mask], res["preds"][m][r][mask], w[mask]) for r in range(N_REPEATS)]
                for k in METS:
                    vals = [a[k] for a in acc]
                    out[(gname, m, k)] = np.nan if all(np.isnan(vals)) else float(np.nanmean(vals))
        return out
    point = one(np.ones(len(y)))
    boot = {}
    t0 = time.time()
    for b, w in enumerate(cluster_weights(g, B, seed)):
        for k, v in one(w).items():
            boot.setdefault(k, []).append(v)
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}

def report_groups(res, point, boot, groups):
    y = res["y"]
    rows = []
    for gname, mask in groups.items():
        n = int(mask.sum()); ev = int(y[mask].sum()); pat = len(set(res["g"][mask]))
        for m in EVAL_MODELS:
            rows.append({"group": gname, "model": m, "adm": n, "patients": pat, "events": ev, "prev": f"{ev/max(n,1):.4f}",
                         "AUROC": _fmt(point[(gname, m, "auroc")]) + _ci(boot[(gname, m, "auroc")]),
                         "AUPRC": _fmt(point[(gname, m, "auprc")]) + _ci(boot[(gname, m, "auprc")]),
                         "O/E": _fmt(point[(gname, m, "oe")], 2) + _ci(boot[(gname, m, "oe")], 2),
                         "slope": _fmt(point[(gname, m, "slope")], 2) + _ci(boot[(gname, m, "slope")], 2)})
    print(pd.DataFrame(rows).to_string(index=False))

In [7]:
# ---- 2/3. temporal and ICD-transition validation: train on one block of calendar time, test on a later one
def temporal_plan(d):
    """Rolling origin by anchor_year_group (train on all earlier groups, test on one later group) and ICD-9 -> ICD-10.
    Patients who appear in the test block are removed from the training block (no patient on both sides)."""
    eidx = d["era"].map({e: i for i, e in enumerate(ERA_ORDER)})
    plan = []
    for k in range(1, len(ERA_ORDER)):
        te = np.where(eidx == k)[0]
        test_pat = set(d["subject_id"].values[te])
        tr = np.where((eidx < k).values & ~d["subject_id"].isin(test_pat).values)[0]
        plan.append((f"rolling: train {ERA_ORDER[0][:4]}-{ERA_ORDER[k-1][-4:]} -> test {ERA_ORDER[k]}", tr, te, f"era {ERA_ORDER[k]}"))
    te = np.where(d["icd_ver"].values == "ICD-10 only")[0]
    test_pat = set(d["subject_id"].values[te])
    tr = np.where((d["icd_ver"].values == "ICD-9 only") & ~d["subject_id"].isin(test_pat).values)[0]
    plan.append(("ICD transition: train ICD-9-only -> test ICD-10-only", tr, te, "ICD coding: ICD-10 only"))
    return plan

def temporal_fit(res, tr, te, seed):
    X, y, g, prep = res["X"], res["y"], res["g"], res["prep"]
    out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, seed)
    return {m: out[m] for m in EVAL_MODELS}, chosen

def temporal_eval(y_te, g_te, preds, B, seed):
    point = {}
    for m in EVAL_MODELS:
        for k, v in perf(y_te, preds[m], np.ones(len(y_te))).items():
            point[(m, k)] = v
    for k in ("auroc", "auprc"):
        point[("Stack-LR", k)] = point[("Stack", k)] - point[("LR", k)]
    boot = {}
    for w in cluster_weights(g_te, B, seed):
        cur = {}
        for m in EVAL_MODELS:
            for k, v in perf(y_te, preds[m], w).items():
                cur[(m, k)] = v
        for k in ("auroc", "auprc"):
            cur[("Stack-LR", k)] = cur[("Stack", k)] - cur[("LR", k)]
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
    return point, {k: np.array(v) for k, v in boot.items()}

def run_temporal(res, B, seed):
    d = res["spec"]["data"]; y = res["y"].astype(float); g = res["g"]
    results = []
    for i, (label, tr, te, grp) in enumerate(temporal_plan(d)):
        info = dict(label=label, group=grp, train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                    test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()))
        if info["train_events"] < MIN_TRAIN_EVENTS or info["test_events"] < MIN_TEST_EVENTS:
            info["skipped"] = True
            print(f"[{label}] SKIPPED (train events {info['train_events']}, test events {info['test_events']})")
            results.append(info); continue
        assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
        t0 = time.time()
        preds, chosen = temporal_fit(res, tr, te, SEED + 500 + i)
        print(f"[{label}] fitted in {(time.time()-t0)/60:.1f} min (train adm {len(tr)}, test adm {len(te)})")
        point, boot = temporal_eval(y[te], g[te], preds, B, SEED + 900 + i)
        info.update(skipped=False, point=point, boot=boot, chosen=chosen)
        results.append(info)
    return results

def report_temporal(results, cv_point, cv_boot):
    rows = []
    for info in results:
        base = {"design": info["label"], "train adm/pat/events": f"{info['train_adm']}/{info['train_pat']}/{info['train_events']}",
                "test adm/pat/events": f"{info['test_adm']}/{info['test_pat']}/{info['test_events']}"}
        if info["skipped"]:
            rows.append({**base, "model": "skipped (too few events)"}); continue
        for m in EVAL_MODELS:
            p, b = info["point"], info["boot"]
            rows.append({**base, "model": m,
                         "AUROC": _fmt(p[(m, "auroc")]) + _ci(b[(m, "auroc")]),
                         "AUPRC": _fmt(p[(m, "auprc")]) + _ci(b[(m, "auprc")]),
                         "O/E": _fmt(p[(m, "oe")], 2) + _ci(b[(m, "oe")], 2),
                         "slope": _fmt(p[(m, "slope")], 2) + _ci(b[(m, "slope")], 2),
                         "CITL": _fmt(p[(m, "citl")], 2) + _ci(b[(m, "citl")], 2),
                         "CV AUROC, same block": _fmt(cv_point.get((info["group"], m, "auroc"), np.nan))})
    print(pd.DataFrame(rows).to_string(index=False))
    print("\n    Paired difference Stack - LR in the later block (95% patient-level cluster-bootstrap interval)")
    rows = []
    for info in results:
        if info["skipped"]:
            continue
        p, b = info["point"], info["boot"]
        rows.append({"design": info["label"],
                     "AUROC diff": f"{p[('Stack-LR','auroc')]:+.3f}" + _ci(b[("Stack-LR", "auroc")]),
                     "AUPRC diff": f"{p[('Stack-LR','auprc')]:+.3f}" + _ci(b[("Stack-LR", "auprc")])})
    print(pd.DataFrame(rows).to_string(index=False))

In [8]:
# ---- 4. refit without / with era-related variables (paired comparison, same splits, same patient resamples)
def paired_alt(res_ref, alts, B, seed):
    """alts = {tag: res}. Returns point estimates, bootstrap draws, and differences (alt - ref)."""
    y = res_ref["y"].astype(float); g = res_ref["g"]
    for tag, r in alts.items():
        assert np.array_equal(r["y"], res_ref["y"]) and np.array_equal(r["g"], res_ref["g"]), "analyses must share rows and patients"
    models = ["LR", "RF", "ET", "XGB", "Stack"]
    mets = ("auroc", "auprc", "slope", "oe")
    allres = {"ref": res_ref, **alts}
    def one(w):
        out = {}
        for tag, res in allres.items():
            for m in models:
                acc = [perf(y, res["preds"][m][r], w) for r in range(N_REPEATS)]
                for k in mets:
                    out[(m, tag, k)] = float(np.mean([a[k] for a in acc]))
        return out
    point = one(np.ones(len(y)))
    boot = {}
    t0 = time.time()
    for b, w in enumerate(cluster_weights(g, B, seed)):
        cur = one(w)
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
        for tag in alts:
            for m in models:
                for k in mets:
                    boot.setdefault((m, f"diff_{tag}", k), []).append(cur[(m, tag, k)] - cur[(m, "ref", k)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}, models

def report_paired(res_ref, alts, point, boot, models):
    y = res_ref["y"]; d = res_ref["spec"]["data"]
    print("main          : numeric", res_ref["spec"]["num"], "binary", res_ref["spec"]["bin"], "categorical", res_ref["spec"]["cat"])
    for tag, r in alts.items():
        print(f"{tag:14s}: numeric", r["spec"]["num"], "binary", r["spec"]["bin"], "categorical", r["spec"]["cat"])
    for met, dd in (("auroc", 3), ("auprc", 3)):
        rows = []
        for m in models:
            r_ = {"model": m, f"{met} main": f"{point[(m,'ref',met)]:.{dd}f}"}
            for tag in alts:
                r_[f"{tag}"] = f"{point[(m,tag,met)]:.{dd}f}"
                r_[f"diff ({tag} - main)"] = f"{point[(m,tag,met)] - point[(m,'ref',met)]:+.{dd}f}" + _ci(boot[(m, f"diff_{tag}", met)], dd)
            rows.append(r_)
        print(f"\n    {met.upper()} (paired, 95% patient-level cluster-bootstrap intervals for the differences)")
        print(pd.DataFrame(rows).to_string(index=False))
    rows = []
    for m in models:
        r_ = {"model": m, "slope main": f"{point[(m,'ref','slope')]:.2f}", "O/E main": f"{point[(m,'ref','oe')]:.2f}"}
        for tag in alts:
            r_[f"slope {tag}"] = f"{point[(m,tag,'slope')]:.2f}"; r_[f"O/E {tag}"] = f"{point[(m,tag,'oe')]:.2f}"
        rows.append(r_)
    print("\n    Calibration after recalibration (slope; observed/expected)")
    print(pd.DataFrame(rows).to_string(index=False))
    for tag, r in [("main", res_ref)] + list(alts.items()):
        ch = pd.DataFrame(r["chosen"], columns=["learner", "config"])
        tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
        print(f"\n    Hyperparameters selected in the inner loops, {tag} (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
        print(tab.to_string(index=False))

In [9]:
def emit(name, fn):
    """Print the aggregate report and also keep it as a text file (aggregates only) so it can be shared without truncation."""
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    print(text)
    path = f"outputs_{name}.txt"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"[aggregate report saved as notebooks/revision/{path}  ({len(text)} characters)]")

### **7. Run the analyses**

Each cell prints the projected time after the first fold.

#### 7.1 `main` (replicate of notebooks 02/04)

In [10]:
res_main = run_cv(make_spec("main"))
sp = res_main["spec"]
print("main predictors:", sp["num"], sp["bin"], sp["cat"])

[main] first outer fold took 0.6 min -> projected total about 9 min
[main] cross-validation finished in 9.2 min; every admission scored exactly once per repeat: True
main predictors: ['age_at_admission', 'hemoglobin', 'creatinine'] ['creatinine_missing'] ['ethnicity', 'insurance', 'language', 'admission_type', 'admission_location']


#### 7.2 `main_noadmin` (without the five administrative categorical variables)

In [11]:
res_noadm = run_cv(make_spec("main_noadmin"))

[main_noadmin] first outer fold took 0.6 min -> projected total about 8 min
[main_noadmin] cross-validation finished in 9.1 min; every admission scored exactly once per repeat: True


#### 7.3 `main_plus_era` (adds the anchor-year group; diagnostic)

In [12]:
res_era = run_cv(make_spec("main_plus_era"))

[main_plus_era] first outer fold took 0.9 min -> projected total about 14 min
[main_plus_era] cross-validation finished in 12.9 min; every admission scored exactly once per repeat: True


#### 7.4 Performance by anchor-year group and ICD coding version (from the out-of-fold predictions of `main`)

In [13]:
d_main = res_main["spec"]["data"]
groups = {"All eligible deliveries": np.ones(len(d_main), bool)}
for e in ERA_ORDER:
    groups[f"era {e}"] = (d_main["era"] == e).values
for v in ["ICD-9 only", "mixed", "ICD-10 only"]:
    groups[f"ICD coding: {v}"] = (d_main["icd_ver"] == v).values
groups = {k: m for k, m in groups.items() if int(m.sum()) > 0}
pt_grp, bt_grp = group_performance(res_main, groups, BOOT_SENS, SEED + 31)

  bootstrap: 5 replicates took 3 s -> projected 3.3 min for B=300


#### 7.5 Rolling-origin temporal validation and ICD-9 to ICD-10 transition

In [14]:
temporal = run_temporal(res_main, BOOT_SENS, SEED + 41)

[rolling: train 2008-2010 -> test 2011 - 2013] fitted in 0.6 min (train adm 8212, test adm 4328)
[rolling: train 2008-2013 -> test 2014 - 2016] fitted in 0.6 min (train adm 12540, test adm 3569)
[rolling: train 2008-2016 -> test 2017 - 2019] fitted in 0.7 min (train adm 16109, test adm 2343)
[rolling: train 2008-2019 -> test 2020 - 2022] fitted in 0.8 min (train adm 18452, test adm 931)
[ICD transition: train ICD-9-only -> test ICD-10-only] fitted in 0.5 min (train adm 8507, test adm 8864)


#### 7.6 Paired comparison of `main` with the two refits

In [15]:
alts = {"no_admin_vars": res_noadm, "plus_era": res_era}
pt_pair, bt_pair, models_pair = paired_alt(res_main, alts, BOOT, SEED + 51)

  bootstrap: 5 replicates took 2 s -> projected 6.0 min for B=1000


### **8. Report**

Prints the aggregate tables and saves them as `outputs_era_temporal.txt`. Paste the text file (or the printed output) back for the analysis.

In [16]:
def full_report():
    y = res_main["y"]; d = res_main["spec"]["data"]
    print("=" * 110)
    print(f"ERA / TEMPORAL SENSITIVITY: SMM, eligible deliveries, n={len(y)}, patients={d.subject_id.nunique()}, "
          f"events={int(y.sum())}, prevalence={y.mean():.4f}")
    print(f"outer {OUTER_FOLDS}x{N_REPEATS}, inner {INNER_FOLDS}, bootstrap B={BOOT_SENS} (groups, temporal) / {BOOT} (paired refits), QUICK={QUICK}")
    print("=" * 110)

    print("\n[0] Eligible deliveries by anchor-year group and by ICD coding version (aggregate counts)")
    rows = []
    for key, lab in (("era", "anchor_year_group"), ("icd_ver", "ICD coding")):
        t = d.groupby(key, dropna=False).agg(adm=("hadm_id", "size"), patients=("subject_id", "nunique"), events=("y", "sum"))
        t["prev"] = (t.events / t.adm).round(4)
        t.insert(0, "variable", lab); t.index.name = "group"
        rows.append(t.reset_index())
    print(pd.concat(rows).to_string(index=False))
    ct = pd.crosstab(d["era"].fillna("missing"), d["icd_ver"])
    print("\n    Admissions by anchor-year group (rows) and ICD coding version (columns)")
    print(ct.to_string())

    print("\n[1] Cross-validated performance by group (out-of-fold predictions of the main analysis; patients are in the same folds as notebooks 02/04)")
    print("    O/E and slope come from probabilities calibrated on the whole development sample, so O/E far from 1 in one group means a calibration shift between groups.")
    report_groups(res_main, pt_grp, bt_grp, groups)

    print("\n[2] Temporal validation (rolling origin) and ICD-9 -> ICD-10 transition: models fitted only on the earlier block, scored on the later one")
    print("    Patients with admissions in the later block are removed from the earlier block. Each model is tuned, stacked and recalibrated inside the earlier block only.")
    report_temporal(temporal, pt_grp, bt_grp)

    print("\n[3] Refits without the administrative variables that can reflect the era, and with the era added as a diagnostic")
    report_paired(res_main, alts, pt_pair, bt_pair, models_pair)

emit("era_temporal", full_report)

ERA / TEMPORAL SENSITIVITY: SMM, eligible deliveries, n=19383, patients=13710, events=445, prevalence=0.0230
outer 5x3, inner 3, bootstrap B=300 (groups, temporal) / 1000 (paired refits), QUICK=False

[0] Eligible deliveries by anchor-year group and by ICD coding version (aggregate counts)
      group          variable   adm  patients  events   prev
2008 - 2010 anchor_year_group  8212      5536     121 0.0147
2011 - 2013 anchor_year_group  4328      2977      86 0.0199
2014 - 2016 anchor_year_group  3569      2507     104 0.0291
2017 - 2019 anchor_year_group  2343      1813      81 0.0346
2020 - 2022 anchor_year_group   931       877      53 0.0569
ICD-10 only        ICD coding  8864      7070     271 0.0306
 ICD-9 only        ICD coding 10519      8285     174 0.0165

    Admissions by anchor-year group (rows) and ICD coding version (columns)
icd_ver      ICD-10 only  ICD-9 only
era                                 
2008 - 2010         1529        6683
2011 - 2013         1510        2